In [ ]:
import os
import sys
import time
import jax
import jax.numpy as jnp
import viser

sys.path.append(os.path.abspath(".."))
from environments import RopeEnv

In [ ]:
server = viser.ViserServer()
_ = server.scene.add_grid(name="ground")

In [ ]:
env = RopeEnv(
    time_step=0.02,       # s
    num_segments=10,
    rope_length=0.2,      # m
    rope_diameter=0.004,  # m
    youngs_modulus=1e5,   # Pa
    mass_density=300,     # kg/m³
    weld_to_rope_ends=True,
    grip_stiffness=100,   # N/m
)

# Specify initial node positions
theta = jnp.deg2rad(20)
l = jnp.arange(env.params.num_nodes) * env.params.segment_length
x_node = jnp.vstack((
    l * jnp.cos(theta),
    jnp.zeros(env.params.num_nodes),
    l * jnp.sin(theta) + 0.01,
)).T

state = env.state(x_node=x_node)
env.visualize(server, state)

In [ ]:
for i in range(99999):
    t = i * env.params.dt

    v_weld = jnp.array([
        [0, 0.2*jnp.cos(2*jnp.pi * t), 0],
        [0.3*jnp.cos(2*jnp.pi * t), 0, 0],
    ])
    control = env.control(v_weld=v_weld)

    start = time.time()
    state, lin = jax.vjp(env.step, state, control)
    A, B = jax.vmap(lin)(jnp.eye(state.size))
    elapsed = time.time() - start
    print(f"\rStep took {elapsed*1e3:.2f} ms  ", end="")

    if 'A' in locals() and jnp.isnan(A).any():
        print(A)
        raise RuntimeError("NaN occurred")
    if 'B' in locals() and jnp.isnan(B).any():
        print(B)
        raise RuntimeError("NaN occurred")

    env.visualize(server, state)

    wait = env.params.dt - elapsed
    if wait > 0:
        time.sleep(wait)